# Chapter 36: Computer Vision Foundations & Image Geometry
## Multimedia & Computer Vision Learning Hub - ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Objective:** Memahami sistem koordinat citra, transformasi geometri homogen 2D, pemetaan mundur (*inverse mapping*), dan interpolasi bilinear.

### 1. Landasan Teori & Formulasi Matematis
Transformasi afinitas pada ruang homogen 2D $\mathbb{P}^2$ dinyatakan sebagai:
$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

Untuk menghindari celah kosong (*voids*), kita menerapkan **pemetaan mundur (*inverse mapping*)**:
$$\mathbf{p} = \mathbf{M}^{-1} \mathbf{p}'$$
Diikuti oleh interpolasi bilinear untuk menghitung intensitas pada koordinat kontinu $(x, y)$.

In [ ]:
# 2. Dependency Setup & Environment
import numpy as np
import cv2
# Note: matplotlib bersifat opsional jika dijalankan di Google Colab / Jupyter Lab
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. Sintesis Dataset Uji (Pola Grid Geometri & Objek Uji)
def generate_test_image(height=200, width=200):
    img = np.zeros((height, width), dtype=np.uint8)
    # Gambar kotak dan garis grid untuk mengamati distorsi geometris
    img[40:160, 40:160] = 120
    img[60:140, 60:140] = 220
    img[::20, :] = 255
    img[:, ::20] = 255
    return img

src_image = generate_test_image()
print(f"Dimensi Citra Uji: {src_image.shape}, Rentang Nilai: [{src_image.min()}, {src_image.max()}]")

In [ ]:
# 4. Implementasi Algoritma: Pemetaan Mundur & Interpolasi Bilinear Manual
def warp_affine_inverse_bilinear(img, M, output_shape):
    out_h, out_w = output_shape
    in_h, in_w = img.shape[:2]
    out_img = np.zeros((out_h, out_w), dtype=np.float64)
    
    # Hitung matriks invers
    M_inv = np.linalg.inv(M)
    
    # Grid koordinat target
    grid_x, grid_y = np.meshgrid(np.arange(out_w), np.arange(out_h))
    ones = np.ones_like(grid_x)
    homo_coords = np.stack([grid_x, grid_y, ones], axis=-1) # (out_h, out_w, 3)
    
    # Pemetaan mundur ke koordinat sumber: p = M_inv * p'
    src_coords = np.dot(homo_coords, M_inv.T) # (out_h, out_w, 3)
    src_x = src_coords[..., 0]
    src_y = src_coords[..., 1]
    
    # Interpolasi bilinear vektor terakselerasi
    x0 = np.floor(src_x).astype(int)
    x1 = x0 + 1
    y0 = np.floor(src_y).astype(int)
    y1 = y0 + 1
    
    # Mask validitas batas citra
    valid_mask = (x0 >= 0) & (x1 < in_w) & (y0 >= 0) & (y1 < in_h)
    
    alpha = src_x - x0
    beta = src_y - y0
    
    # Evaluasi nilai piksel
    val = (
        (1 - alpha) * (1 - beta) * np.where(valid_mask, img[np.clip(y0, 0, in_h-1), np.clip(x0, 0, in_w-1)], 0) +
        alpha * (1 - beta) * np.where(valid_mask, img[np.clip(y0, 0, in_h-1), np.clip(x1, 0, in_w-1)], 0) +
        (1 - alpha) * beta * np.where(valid_mask, img[np.clip(y1, 0, in_h-1), np.clip(x0, 0, in_w-1)], 0) +
        alpha * beta * np.where(valid_mask, img[np.clip(y1, 0, in_h-1), np.clip(x1, 0, in_w-1)], 0)
    )
    
    out_img = np.where(valid_mask, val, 0)
    return np.clip(out_img, 0, 255).astype(np.uint8)

# Buat matriks transformasi afinitas (Rotasi 30 derajat + Skala 0.8 + Translasi [20, 10])
angle_rad = np.radians(30.0)
M_test = np.array([
    [0.8 * np.cos(angle_rad), -0.8 * np.sin(angle_rad), 20.0],
    [0.8 * np.sin(angle_rad),  0.8 * np.cos(angle_rad), 10.0],
    [0.0,                      0.0,                     1.0]
])

warped_manual = warp_affine_inverse_bilinear(src_image, M_test, (200, 200))
# Bandingkan dengan fungsi bawaan OpenCV
warped_opencv = cv2.warpAffine(src_image, M_test[:2], (200, 200), flags=cv2.INTER_LINEAR)

diff = np.abs(warped_manual.astype(float) - warped_opencv.astype(float))
print(f"Mean Absolute Error (Manual vs OpenCV): {np.mean(diff):.4f} px")
print(f"Max Absolute Error: {np.max(diff):.4f} px")
assert np.mean(diff) < 2.0, "Implementasi manual bilinear konsisten dengan standar OpenCV"

In [ ]:
# 5. Evaluasi Kuantitatif & Pembuktian Sifat Invarian Afinitas
# Sifat afinitas menjaga perbandingan rasio paralelisme
p1 = np.array([20.0, 20.0, 1.0])
p2 = np.array([60.0, 20.0, 1.0])
p3 = np.array([20.0, 50.0, 1.0])
p4 = np.array([60.0, 50.0, 1.0])

# Garis p1-p2 sejajar dengan p3-p4 (gradien horizontal = 0)
tp1, tp2 = M_test @ p1, M_test @ p2
tp3, tp4 = M_test @ p3, M_test @ p4

v12 = tp2[:2] - tp1[:2]
v34 = tp4[:2] - tp3[:2]

# Cross product vektor arah 2D (a_x * b_y - a_y * b_x) harus mendekati 0 untuk garis sejajar
parallel_cross = v12[0] * v34[1] - v12[1] * v34[0]
print(f"Cross product vektor arah tertransformasi: {parallel_cross:.6e}")
assert np.isclose(parallel_cross, 0.0, atol=1e-8), "Sifat paralelisme afinitas terbukti invarian"

### 6. Batasan & Analisis Kegagalan (Failure Modes)
1. **Singularitas Determinan**: Bila $\det(\mathbf{M}) = 0$, matriks tidak dapat diinversikan. Periksa nilai kondisi (*condition number*).
2. **Aliasing pada Magnifikasi Tinggi**: Pemetaan mundur pada skala sangat kecil menyebabkan hilangnya informasi frekuensi tinggi (butuh antialiasing filter).
3. **Distorsi Ekstrem Perspektif**: Homografi dengan titik lenyap di dalam bidang citra memicu pembagian dengan nilai mendekati nol ($w \to 0$).

### 7. Referensi Akademik
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications*. Springer.
- Hartley, R., & Zisserman, A. (2004). *Multiple View Geometry in Computer Vision*. Cambridge University Press.